In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('soil-grain-size-from-photos')

print("Path to competition files:", path)

c:\Users\axeld\VSCode\Kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: C:\Users\axeld\.cache\kagglehub\competitions\soil-grain-size-from-photos


In [2]:
# import random
# import matplotlib.pyplot as plt
# from pathlib import Path
# from PIL import Image
# 
# image_dir = Path(f"{path}/Training-All_Photos_updated/Training-All_Photos_updated")
# 
# image_paths = [
#     p for p in image_dir.rglob("*") if p.suffix.lower() in set([".jpg", ".jpeg", ".png", ".bpm", ".webp"])
# ]
# 
# print(f"{len(image_paths)} images in {image_dir.resolve()}")
# 
# sample = random.sample(image_paths, min(9, len(image_paths)))
# 
# if sample:
#     fig, axes = plt.subplots(3, 3, figsize=(12, 12))
# 
#     for ax in axes.flat:
#         ax.axis("off")
# 
#     for ax, image_path in zip(axes.flat, sample):
#         with Image.open(image_path) as img:
#             ax.imshow(img.convert("RGB"))
#         ax.set_title(image_path.name, fontsize=9)
# 
#     plt.tight_layout()
#     plt.show()

In [3]:
import pandas as pd
from pathlib import Path

data_dir = Path(path)

train_labels = pd.read_csv(data_dir / "Training_labels_updated.csv")
train_image_dir = data_dir / "Training-All_Photos_updated/Training-All_Photos_updated"

image_paths = sorted(
    p for p in train_image_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

images = pd.DataFrame({
    "image_path": image_paths,
    "sample_id": [p.stem.rsplit("_", 2)[-2] for p in image_paths]
})

image_table = images.merge(
    train_labels,
    on="sample_id",
    how="left",
    validate="many_to_one",
    indicator=True
)
image_table = image_table.drop(columns="_merge")

target_cols = [c for c in train_labels.columns if c != "sample_id"]

print(f"Target cols: {target_cols}")

image_table.head()


Target cols: ['0.002', '0.0063', '0.02', '0.063', '0.2', '0.63', '2', '6.3', '20', '63', '200']


,image_path,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
1,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
2,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
3,C:\Users\axeld\.cache\kagglehub\competitions\s...,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0
4,C:\Users\axeld\.cache\kagglehub\competitions\s...,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0


In [6]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)

train_idx, val_idx = next(
    splitter.split(image_table, groups=image_table["sample_id"])
)

print(f"train_idx: {train_idx}")
print(f"val_idx: {val_idx}")

train_table = image_table.iloc[train_idx].reset_index(drop=True)
val_table = image_table.iloc[val_idx].reset_index(drop=True)

train_idx: [  0   1   2   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19
  20  21  22  23  24  25  26  27  32  33  34  35  36  37  38  42  43  44
  45  46  47  48  49  50  51  55  56  57  60  61  62  63  64  65  66  67
  68  69  70  71  75  76  77  78  79  80  81  82  83  84  85  86  87  88
  89  90  91  92  93  94  95  96 101 102 103 104 105 106 107 109 110 114
 115 116 117 118 119 120 121 122 123 124 125 126]
val_idx: [  3   4  28  29  30  31  39  40  41  52  53  54  58  59  72  73  74  97
  98  99 100 108 111 112 113]


In [5]:
import torch
from PIL import Image
from torch.utils.data import Dataset

class SoilDataset(Dataset):
    def __init__(self, table, target_cols, transform):
        self.table = table.reset_index(drop=True)
        self.targets = torch.tensor(
            self.table[target_cols].to_numpy(dtype="float32") / 100.0
        )
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx):
        with Image.open(self.table.loc[idx, "image_path"]) as image:
            image = self.transform(image.convert("RGB"))

        return image, self.targets[idx]

In [7]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = SoilDataset(train_table, target_cols, transform)
val_dataset = SoilDataset(val_table, target_cols, transform)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

images_batch, targets_batch = next(iter(train_loader))

print(f"images batch shape: {images_batch.shape}")
print(f"targets batch shape: {targets_batch.shape}")

images batch shape: torch.Size([32, 3, 224, 224])
targets batch shape: torch.Size([32, 11])
